In [52]:
import tensorflow_datasets as tfds
import pandas
from tfdatacompose import Pipeline, Take, Print, PrintShape, LambdaMap, Map, DatasetOperation

In [12]:
train = tfds.load('smartwatch_gestures', split='train')
train

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/3251 [00:00<?, ? examples/s]

Shuffling /home/francois/tensorflow_datasets/smartwatch_gestures/1.0.0.incompleteL7TCW2/smartwatch_gestures-tr…

Dataset smartwatch_gestures downloaded and prepared to /home/francois/tensorflow_datasets/smartwatch_gestures/1.0.0. Subsequent calls will reuse this data.


2025-03-13 23:33:29.235851: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-03-13 23:33:29.239924: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


<_PrefetchDataset element_spec={'attempt': TensorSpec(shape=(), dtype=tf.uint8, name=None), 'features': {'accel_x': TensorSpec(shape=(None,), dtype=tf.float64, name=None), 'accel_y': TensorSpec(shape=(None,), dtype=tf.float64, name=None), 'accel_z': TensorSpec(shape=(None,), dtype=tf.float64, name=None), 'time_event': TensorSpec(shape=(None,), dtype=tf.uint64, name=None), 'time_millis': TensorSpec(shape=(None,), dtype=tf.uint64, name=None), 'time_nanos': TensorSpec(shape=(None,), dtype=tf.uint64, name=None)}, 'gesture': TensorSpec(shape=(), dtype=tf.int64, name=None), 'participant': TensorSpec(shape=(), dtype=tf.uint8, name=None)}>

In [61]:
class ExtractData(Map):
    def map(self, e):
        return e['features'], e['gesture']

class ExtractAccelerometers(Map):
    def map(self, features, gesture):
        return [features['accel_x'], features['accel_y'], features['accel_z']], gesture

class ConcatenateAccelerometers(Map):
    def map(self, features, gesture):
        return features[0] + features[1] + features[2]

class ScaleAccelerometers(Map):
    def __init__(self, mu, sigma):
        self.mu = mu
        self.sigma = sigma
        
    def map(self, features, gesture):
        scaled = (features - self.mu)/self.sigma
        return scaled, gesture

data = Pipeline([
    Extract(),
    MergeAccelerometers(),
    ScaleAccelerometers(1, 10),
])(train)
list(data.as_numpy_iterator())

[(array([[-0.1153229 , -0.1306458 , -0.0540313 , -0.1612916 , -0.0233855 ,
           0.20645781,  0.17581201,  0.23710361,  0.25242651,  0.31371808,
           0.25242651],
         [-0.0540313 , -0.1       , -0.0693542 , -0.1612916 , -0.1766145 ,
          -0.0233855 , -0.0846771 , -0.1       , -0.1306458 , -0.1766145 ,
          -0.1306458 ],
         [ 0.92663374,  0.81937342,  0.2677494 ,  0.2983952 ,  0.58953009,
           1.47825785,  1.35567455,  1.14115419,  1.04921684,  0.78872766,
           0.85001926]]),
  3),
 (array([[ 0.22178071,  0.23710361,  0.037906  ,  0.0685518 ,  0.45162411,
           0.22178071, -0.0080627 , -0.0846771 ,  0.17581201,  0.55888429,
           0.2830723 ,  0.20645781,  0.22178071,  0.23710361,  0.23710361,
           0.20645781],
         [-0.0387084 , -0.0846771 , -0.1459687 , -0.0693542 , -0.237906  ,
          -0.2838747 ,  0.0532289 , -0.2072602 , -0.54436378, -0.75888429,
          -0.2225831 , -0.37581201, -0.529041  , -0.43710361, -0.513718

In [ ]:
[(array([[-0.153229  , -0.306458  ,  0.45968699, -0.61291599,  0.76614499,
           3.06457806,  2.75812006,  3.37103605,  3.52426505,  4.13718081,
           3.52426505],
         [ 0.45968699,  0.        ,  0.306458  , -0.61291599, -0.76614499,
           0.76614499,  0.153229  ,  0.        , -0.306458  , -0.76614499,
          -0.306458  ],
         [10.26633739,  9.19373417,  3.67749405,  3.98395205,  6.89530087,
          15.78257847, 14.55674553, 12.41154194, 11.49216843,  8.88727665,
           9.50019264]]),
  3),